In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/intentmap-nids
!ls

Mounted at /content/drive
/content/drive/MyDrive/intentmap-nids
audit_duplicates.py  models		train_models.py
build_pipeline.py    profile_unsw.py	UNSW_NB15_testing-set.csv
config		     reports		UNSW_NB15_training-set.csv
data		     sanity_check.py
evidence.zip	     shift_analysis.py


In [ ]:
import pandas as pd
train = pd.read_csv('data/raw/UNSW_NB15_training-set.csv')
test  = pd.read_csv('data/raw/UNSW_NB15_testing-set.csv')
print(train.shape, test.shape)
print(train.columns.tolist())

(175341, 45) (82332, 45)
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']


In [ ]:
!python profile_unsw.py --data-dir data/raw --out reports
!python audit_duplicates.py --data-dir data/raw --out reports

Wrote reports/data_quality_report.md
Wrote reports/feature_separability.csv
Wrote 2 figures to reports
Wrote reports/duplicate_audit.md

  Test rows overlapping training:        8,541 (10.37%)
  Normal test rows leaking from training: 1,193 (3.22%)   <-- the number that matters


In [ ]:
!python build_pipeline.py --data-dir data/raw

Train      (41065, 41)
Validation (10266, 41)
Test full  (82332, 41)   dedup (80995, 41)
Output columns: 41

Wrote models/preprocessor.joblib, config/feature_order.json,
      config/feature_dictionary.csv, reports/pipeline_report.md


In [ ]:
!python sanity_check.py

train (41065, 41)  val (10266, 41)  test (82332, 41)
fitting Isolation Forest on normal training rows only ...

  ROC-AUC        0.795
  PR-AUC         0.845
  effect size    1.24

  budget  0.5%   precision 0.976   recall 0.234   F1 0.378   actual FPR 0.70%
  budget  1.0%   precision 0.942   recall 0.282   F1 0.434   actual FPR 2.13%
  budget  3.0%   precision 0.880   recall 0.592   F1 0.708   actual FPR 9.90%

Wrote reports/sanity_check.md and reports/sanity_check_scores.png


In [ ]:
!python shift_analysis.py

  adversarial AUC            0.641
  features with PSI > 0.25   6 of 41
  most shifted feature       dload (PSI 0.394)

Wrote reports/shift_analysis.md, reports/feature_shift.csv,
      reports/distribution_shift.png


In [ ]:
!ls -lh reports/ config/ models/ data/processed/

config/:
total 8.5K
-rw------- 1 root root 2.7K Sep 19 13:55 feature_dictionary.csv
-rw------- 1 root root 1.3K Sep 19 13:55 feature_order.json
-rw------- 1 root root  827 Aug 31 00:54 scaling_repair.json
-rw------- 1 root root 2.9K Aug 31 01:00 threshold_config.json

data/processed/:
total 8.3M
-rw------- 1 root root 2.8M Sep 19 13:55 test_dedup.npz
-rw------- 1 root root 2.8M Sep 19 13:55 test_full.npz
-rw------- 1 root root 2.3M Sep 19 13:55 train_normal.npz
-rw------- 1 root root 579K Sep 19 13:55 val_normal.npz

models/:
total 33M
-rw------- 1 root root  87K Aug 31 01:00 autoencoder.keras
-rw------- 1 root root 2.9M Aug 31 01:00 iforest.joblib
-rw------- 1 root root 2.3M Sep 19 13:56 iforest_sanity.joblib
-rw------- 1 root root  19M Aug 31 01:00 lof.joblib
-rw------- 1 root root 5.1K Sep 19 13:55 preprocessor.joblib
-rw------- 1 root root 8.3M Aug 31 01:00 scores.npz

reports/:
total 352K
-rw------- 1 root root  78K Sep 19 13:54 core_feature_distributions.png
-rw------- 1 root roo

In [ ]:
import json
cfg = json.load(open('config/feature_order.json'))
print('columns:', cfg['n_features_out'])
print(cfg['output_columns'])

columns: 41
['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'smean', 'dmean', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_sm_ips_ports', 'proto_arp', 'proto_other', 'proto_tcp', 'proto_udp', 'service_-', 'service_dns', 'service_ftp', 'service_ftp-data', 'service_http', 'service_pop3', 'service_radius', 'service_smtp', 'service_snmp', 'service_ssh', 'state_CON', 'state_ECO', 'state_FIN', 'state_INT', 'state_PAR', 'state_REQ', 'state_RST', 'state_URN', 'state_no']


In [ ]:
import pandas as pd
pd.read_csv('config/feature_dictionary.csv').head(25)

,unsw_feature,status,zeek_source,in_portable_set
0,dur,Direct,duration,yes
1,spkts,Direct,orig_pkts,yes
2,dpkts,Direct,resp_pkts,yes
3,sbytes,Direct,orig_ip_bytes (NOT orig_bytes: UNSW includes h...,yes
4,dbytes,Direct,resp_ip_bytes (NOT resp_bytes: UNSW includes h...,yes
5,rate,Derived,(orig_pkts + resp_pkts) / duration,yes
6,sload,Derived,orig_ip_bytes * 8 / duration,yes
7,dload,Derived,resp_ip_bytes * 8 / duration,yes
8,smean,Derived,orig_ip_bytes / orig_pkts,yes
9,dmean,Derived,resp_ip_bytes / resp_pkts,yes
